# 第7讲：LLaVA-Next 多模态视觉分析——面向数字人文低质量影像的生成式理解

第2讲中，我们用 CLIP 将视频帧与古籍文本映射到同一个嵌入空间，计算相似度——但 CLIP 只能给出「这张图和这段文字有多像」的数字，无法生成任何语言描述。本讲向前迈出关键一步：引入 **LLaVA-Next**（LLaVA-1.6），一个真正意义上的**生成式多模态大模型**。LLaVA-Next 将图像直接注入语言模型的上下文，使模型既能「看见」画面，又能用自然语言回答关于它的任何问题——文字识别、文物描述、档案编目，在同一次推理中一气呵成。这对数字人文中大量存在的低分辨率扫描件、模糊历史影像、文物残片照片，具有直接的实用价值。

# 本讲结构

## 1  环境准备与 GPU 自检
## 2  LLaVA-Next 与 CLIP（第2讲）的关键差异
## 3  加载 LLaVA-Next（4 位量化，Mistral-7B 骨干）
## 4  检查点：确认模型可处理图像
## 5  上传真实数字人文影像帧
## 6  数字人文分析任务一：文字识别与转录
## 7  数字人文分析任务二：文化元素识别
## 8  数字人文分析任务三：标准化档案描述
## 9  解读分析结果
## 10  可复用分析函数
## 快速回顾

# 透明度说明

> **模型来源**：本讲使用 `llava-hf/llava-v1.6-mistral-7b-hf`，由 Hugging Face 社区托管，可直接下载，**无需账号或访问令牌**。
>
> **架构简述**：LLaVA-Next 由三个组件构成——
> - **视觉编码器**：CLIP ViT-L/14（冻结），将图像转为高维视觉特征
> - **投影层（MLP Connector）**：可训练的两层全连接网络，将视觉特征映射为语言模型可读的 token 嵌入
> - **语言模型**：Mistral-7B-Instruct，负责基于融合后的视觉+文本 token 生成回答
>
> 相比 LLaVA-1.5，LLaVA-Next 引入了**动态高分辨率切片**机制：将输入图像切分为多个子块分别编码，再拼合为完整特征序列，使模型能更好地处理低质量或高密度文字影像。

# 第1章  环境准备与 GPU 自检

- 安装四个关键库：`transformers`（模型）、`accelerate`（多设备调度）、`bitsandbytes`（4 位量化）、`Pillow`（图像处理）
- LLaVA-Next 加载 4 位量化后约需 **12–14 GB 显存**，必须使用 T4 或更高规格的 GPU
- 本讲的 GPU 自检采用**硬性失败**策略——未检测到 GPU 时直接抛出异常，避免模型在 CPU 上静默 OOM

> **本地用户（mllm-dh 环境）**：如果已在 `mllm-dh` conda 环境中运行，可注释掉 `!pip install` 行，直接执行 GPU 自检。

In [ ]:
# ===== 第1章：环境准备 =====
# Colab 用户：运行此单元格自动安装依赖。
# 本地 mllm-dh 环境用户：可注释掉 !pip 行，直接运行 GPU 自检。

!pip install -q transformers accelerate bitsandbytes Pillow

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        '未检测到 GPU。请在 Colab 中选择：运行时 → 更改运行时类型 → T4 GPU。'
        ' LLaVA-Next 加载 4 位量化后约需 12–14 GB 显存，无法在 CPU 上运行。'
    )

device_name = torch.cuda.get_device_name(0)
total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU 已就绪：{device_name}')
print(f'显存总量：{total_mem:.1f} GB')

# 第2章  LLaVA-Next 与 CLIP（第2讲）的关键差异

| 维度 | 第2讲 CLIP | 第7讲 LLaVA-Next |
|------|-----------|------------------|
| **输出类型** | 相似度分数（标量） | 自由文本（任意长度） |
| **图像处理方式** | 图像→嵌入向量，与文本嵌入对齐 | 图像→视觉 token，直接注入 LLM 上下文 |
| **能做什么** | 跨模态检索、匹配、排序 | 描述、识别、转录、问答、档案编目 |
| **无法做什么** | 生成任何文字；理解复杂关系 | 不适合做相似度排序（无固定分数） |

## 对数字人文的关键意义

CLIP 告诉你「这帧画面和这段文字有 0.82 的相似度」；  
LLaVA-Next 则能告诉你「画面左上角有'大明万历'四字，右下方可见一件青铜器，器身饰有饕餮纹」。

对于低分辨率扫描件、模糊历史影像、残损文物照片，LLaVA-Next 的**动态高分辨率切片**机制允许模型对局部细节单独编码，这正是 CLIP 单一嵌入策略难以企及的。

# 第3章  加载 LLaVA-Next（4 位量化，Mistral-7B 骨干）

LLaVA-Next 使用**专用的处理器和模型类**，与第1/3/4讲的纯文本模型不同：

| 组件 | 纯文本模型（第1/3/4讲） | LLaVA-Next（本讲） |
|------|------------------------|-------------------|
| 处理器 | `AutoTokenizer` | `LlavaNextProcessor`（同时处理图像和文本） |
| 模型类 | `AutoModelForCausalLM` | `LlavaNextForConditionalGeneration` |
| 推理输入 | 仅文本 token | 文本 token + PIL Image 对象 |

4 位量化配置与第3讲完全一致（NF4 + 双重量化 + bf16/fp16 计算精度），这是在 T4 上运行 7B 视觉语言模型的标准做法。

In [ ]:
# ===== 第3章：量化配置与处理器加载 =====
# 与第3讲的 BitsAndBytesConfig 完全一致，此为在 T4 上运行 7B 模型的标准配置

from transformers import BitsAndBytesConfig, LlavaNextProcessor, LlavaNextForConditionalGeneration

MODEL_ID = 'llava-hf/llava-v1.6-mistral-7b-hf'

# 根据 GPU 能力选择计算精度（T4 及以下用 fp16；Ampere 及以上用 bf16）
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'计算精度：{compute_dtype}')

# 4 位 NF4 量化配置——与第3讲相同，显存从约 28 GB 压缩至约 5–6 GB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

# LlavaNextProcessor 同时负责图像预处理和文本分词
processor = LlavaNextProcessor.from_pretrained(MODEL_ID)
print('处理器加载完成')

In [ ]:
# ===== 第3章（续）：加载 LLaVA-Next 模型 =====
# 首次运行需从 Hugging Face 下载约 4 GB 权重，Colab 约需 3–5 分钟，请耐心等待
# 无需 Hugging Face 账号或访问令牌

model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model.eval()

print('模型加载完成')
total_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f'参数量：{total_params:.2f} B')
print(f'显存占用（加载后）：{torch.cuda.memory_allocated() / 1e9:.1f} GB')

# 第4章  检查点：确认模型可处理图像

在上传任何真实数字人文素材之前，先用一张**程序生成的占位图像**做一次最小化推理验证。

这是一个**有意设置的检查点**：如果模型权重损坏、量化参数不兼容或显存不足，这里会立即报错——避免等到处理珍贵历史影像时才发现问题。占位图像使用 PIL 直接生成，无需任何外部文件。

In [ ]:
# ===== 检查点 1：用占位图像确认模型可正常推理 =====
# 占位图像完全由 PIL 程序生成，无需上传任何文件

from PIL import Image, ImageDraw
import numpy as np

# 生成一张带有简单几何图形的测试图像（模拟低质量扫描件的灰度外观）
placeholder = Image.new('RGB', (336, 336), color=(240, 235, 220))
draw = ImageDraw.Draw(placeholder)
draw.rectangle([40, 40, 296, 296], outline=(80, 60, 40), width=3)
draw.line([40, 168, 296, 168], fill=(120, 100, 80), width=2)
draw.text((80, 140), 'DH TEST', fill=(60, 40, 20))

# LLaVA-Next 的 chat template 要求使用特定的 [INST] 格式
conversation = [
    {
        'role': 'user',
        'content': [
            {'type': 'image'},
            {'type': 'text', 'text': 'What do you see in this image?'},
        ],
    }
]

prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
inputs = processor(images=placeholder, text=prompt_text, return_tensors='pt').to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
    )

# 仅解码新生成的 token，排除输入 prompt
new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
response = processor.decode(new_tokens, skip_special_tokens=True)

print('===== 检查点 1 输出 =====')
print(f'模型回答：{response}')
print('=' * 30)
print('LLaVA-Next 加载正常，可以继续使用真实数字人文素材。')

# 关于占位图像的说明

检查点使用的是程序生成的几何图形，**并非真实数字人文内容**。后续三个分析任务（文字识别、文化元素识别、档案描述）需要一张来自实际 DH 项目的低质量视频帧。

## 在 Colab 中上传真实帧

适合本讲的素材类型：
- 古籍或碑帖的低分辨率扫描件
- 文物（青铜器、陶瓷、玉器等）的历史档案照片
- 纪录片或文化遗产视频中的模糊截帧

支持 **jpg、png、webp** 等常见格式。下一单元格将自动处理 Colab 上传与本地路径两种场景。

In [ ]:
# ===== 第5章：上传或加载真实数字人文影像帧 =====
# 支持 Colab 上传和本地路径两种方式
# 任何常见图像格式均可（jpg、png、webp 等）

import os
from IPython.display import display as ipy_display

# ---- 本地用户：将此处改为你的图像文件路径 ----
LOCAL_IMAGE_PATH = 'dh_frame.jpg'

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

dh_image = None

if IN_COLAB:
    print('请在弹出的文件选择器中上传一张数字人文影像帧...')
    uploaded = files.upload()
    if uploaded:
        uploaded_name = list(uploaded.keys())[0]
        dh_image = Image.open(uploaded_name).convert('RGB')
        print(f'已上传：{uploaded_name}，尺寸：{dh_image.size}')
    else:
        print('未选择文件，将使用检查点占位图像。')
        dh_image = placeholder
elif os.path.exists(LOCAL_IMAGE_PATH):
    dh_image = Image.open(LOCAL_IMAGE_PATH).convert('RGB')
    print(f'已加载本地图像：{LOCAL_IMAGE_PATH}，尺寸：{dh_image.size}')
else:
    print(f'未找到 {LOCAL_IMAGE_PATH}，将使用检查点占位图像。')
    print('如需使用真实素材，请将 LOCAL_IMAGE_PATH 改为正确路径后重新运行此单元格。')
    dh_image = placeholder

print('预览图像：')
ipy_display(dh_image.resize((400, int(400 * dh_image.height / dh_image.width))))

# 第6–8章  三项数字人文分析任务

本讲对同一张影像依次运行三个提示词，每个提示词对应一种数字人文场景：

| 任务 | 提示词语言 | 目标场景 |
|------|-----------|----------|
| **文字识别与转录** | 中文 | OCR 式文字提取，处理模糊或残损字形 |
| **文化元素识别** | 中文 | 文物材质、形制、纹饰的结构化描述 |
| **标准化档案描述** | 中文 | 可直接入库的元数据记录生成 |

对同一图像反复提问是 VLM 与传统 OCR/图像分类管线的核心区别——**无需为每种任务训练独立模型**，只需改变提示词即可切换分析目标。

In [ ]:
# ===== 辅助函数：单次推理 =====
# 封装 LLaVA-Next 的 chat template + 推理流程，后续三个任务直接调用

def run_llava(image, prompt, max_new_tokens=300):
    """向 LLaVA-Next 发送一张图像和一段提示词，返回模型生成的文本。"""
    conversation = [
        {
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': prompt},
            ],
        }
    ]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(images=image, text=prompt_text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()


print('推理辅助函数已定义，开始运行三项分析任务。')

# 第6章  任务一：文字识别与转录

数字人文影像中最核心的诉求之一是**从低质量图像中提取文字**。LLaVA-Next 的动态切片机制允许它对局部区域单独编码，相比全局嵌入更适合高密度文字扫描件。

提示词要求模型明确标注不确定的字符，这一行为对档案工作至关重要——**「[不确定]」比错误的确定性识别更有价值**。

In [ ]:
# ===== 第6章：任务一——文字识别与转录 =====
# 核心数字人文任务：从低质量历史影像中提取可见文字

PROMPT_TEXT = (
    '请仔细观察图像，识别并转录所有可见的文字或字符。'
    '如果图像质量较低或文字模糊，请尽力辨认并用方括号标注不确定的部分，例如「[不确定]」。'
    '按照文字在图像中的位置顺序（从上到下、从左到右）进行转录。'
)

print('正在运行文字识别任务...')
result_text = run_llava(dh_image, PROMPT_TEXT)

print()
print('===== 任务一：文字识别结果 =====')
print(result_text)
print('=' * 35)

# 第7章  任务二：文化元素识别

除了文字，数字人文影像中往往包含**文物器物、图案纹饰、历史场景**等视觉信息，这些是 OCR 工具完全无法捕捉的内容。

本提示词引导模型从**材质、形制、纹饰**三个维度进行结构化描述，这一框架与国内文物档案的标准著录字段直接对应。

In [ ]:
# ===== 第7章：任务二——文化元素识别 =====
# 提取文字之外的文化和历史视觉信息：材质、形制、纹饰

PROMPT_CULTURAL = (
    '请描述图像中可见的文化元素、文物或历史物品。'
    '包括以下可辨认的特征：'
    '（1）材质（如青铜、陶瓷、纸张、石材等）；'
    '（2）形制（器形、尺寸比例、结构特征）；'
    '（3）纹饰（图案类型、分布位置、工艺手法）。'
    '如有不确定之处，请明确说明。'
)

print('正在运行文化元素识别任务...')
result_cultural = run_llava(dh_image, PROMPT_CULTURAL)

print()
print('===== 任务二：文化元素识别结果 =====')
print(result_cultural)
print('=' * 35)

# 第8章  任务三：标准化档案描述

前两个任务提取的是**原始信息**；本任务要求模型将这些信息**整合为一段可直接入库的档案元数据记录**，格式参照数字人文影像元数据著录的基本字段结构。

提示词明确指定了四个输出字段：内容概述、历史文化元素、图像质量评估、建议分类标签——这使得模型输出可以直接对接后续的元数据解析脚本。

In [ ]:
# ===== 第8章：任务三——标准化档案描述 =====
# 将视觉分析整合为结构化元数据记录，可直接用于档案系统入库

PROMPT_ARCHIVAL = (
    '请为这张图像生成一段标准化的档案描述，包含以下四个部分：\n'
    '1. 【图像内容概述】：简要说明图像的主要内容和视觉焦点（2–3句）\n'
    '2. 【历史与文化元素】：列出可辨认的历史或文化相关元素（文字、文物、场景、符号等）\n'
    '3. 【图像质量评估】：评估图像的清晰度、完整性及对档案工作的影响\n'
    '4. 【建议档案分类标签】：提供3–5个可用于检索的关键词标签'
)

print('正在生成标准化档案描述...')
result_archival = run_llava(dh_image, PROMPT_ARCHIVAL, max_new_tokens=400)

print()
print('===== 任务三：标准化档案描述 =====')
print(result_archival)
print('=' * 35)

# 第9章  解读分析结果

## 什么样的输出是有效的？

| 任务 | 强输出特征 | 弱输出或需警惕的信号 |
|------|-----------|---------------------|
| 文字识别 | 具体字符被命名；不确定部分用方括号标注 | 仅描述「图像中有文字」而不转录；无任何不确定标注 |
| 文化元素 | 具体器形/纹饰名称；明确的材质判断 | 泛化描述（「一件古代器物」）；无法说明具体文化特征 |
| 档案描述 | 四个字段均有实质内容；标签词具体可检索 | 字段内容重复；标签过于宽泛（「历史」、「文物」） |

## 幻觉风险

视觉语言模型（VLM）存在**幻觉**风险——模型可能描述图像中并不存在的内容，尤其在图像质量极差时。本讲使用的 LLaVA-Next 经过指令微调，幻觉率相对可控，但对于档案工作，**所有输出均应与原始影像人工核对后方可入库**。这不是模型的缺陷，而是生成式 AI 在高精度学术场景中的固有局限——了解边界，才能正确使用。

# 第10章  可复用批处理分析函数

将三项分析任务封装为单一函数，使其可以批量应用于多帧视频或整个影像集合。

函数设计要点：
- 通过 `tasks` 参数选择运行哪些分析（适配不同的时间预算）
- 返回字典，便于后续序列化为 JSON 档案记录
- 可直接接入第5讲提取的视频帧列表，构成完整的帧级档案管线

In [ ]:
# ===== 第10章：可复用批处理分析函数 =====
# 封装三项任务，供多帧批处理或第10讲 capstone 管线复用

def analyze_dh_frame(image, tasks=None):
    """
    对单张数字人文影像帧运行指定的分析任务。

    参数
    ----
    image : PIL.Image
        待分析的图像（任意尺寸，RGB 模式）
    tasks : list of str, optional
        要运行的任务列表，可选 'text'、'cultural'、'archival'。
        默认全部运行。

    返回
    ----
    dict
        键为任务名称，值为模型生成的文本。
    """
    if tasks is None:
        tasks = ['text', 'cultural', 'archival']

    prompts = {
        'text': (
            '请仔细观察图像，识别并转录所有可见的文字或字符。'
            '如果图像质量较低或文字模糊，请尽力辨认并用方括号标注不确定的部分，例如「[不确定]」。'
            '按照文字在图像中的位置顺序（从上到下、从左到右）进行转录。'
        ),
        'cultural': (
            '请描述图像中可见的文化元素、文物或历史物品。'
            '包括以下可辨认的特征：'
            '（1）材质（如青铜、陶瓷、纸张、石材等）；'
            '（2）形制（器形、尺寸比例、结构特征）；'
            '（3）纹饰（图案类型、分布位置、工艺手法）。'
            '如有不确定之处，请明确说明。'
        ),
        'archival': (
            '请为这张图像生成一段标准化的档案描述，包含以下四个部分：\n'
            '1. 【图像内容概述】：简要说明图像的主要内容和视觉焦点（2–3句）\n'
            '2. 【历史与文化元素】：列出可辨认的历史或文化相关元素（文字、文物、场景、符号等）\n'
            '3. 【图像质量评估】：评估图像的清晰度、完整性及对档案工作的影响\n'
            '4. 【建议档案分类标签】：提供3–5个可用于检索的关键词标签'
        ),
    }

    max_tokens = {'text': 300, 'cultural': 300, 'archival': 400}

    results = {}
    for task in tasks:
        if task not in prompts:
            print(f'未知任务：{task}，已跳过。')
            continue
        print(f'  运行任务：{task}...')
        results[task] = run_llava(image, prompts[task], max_new_tokens=max_tokens[task])

    return results


# 演示：对已上传的 dh_image 仅运行文字识别和档案描述（节省时间）
print('演示批处理函数（运行文字识别 + 档案描述）：')
demo_results = analyze_dh_frame(dh_image, tasks=['text', 'archival'])
for task, output in demo_results.items():
    print(f'\n--- {task} ---')
    print(output[:200] + ('...' if len(output) > 200 else ''))

# 快速回顾（Quick Check）

- **LLaVA-Next 是什么**：将 CLIP ViT-L 视觉编码器与 Mistral-7B 语言模型通过投影层连接的生成式多模态模型；相比 LLaVA-1.5，引入动态高分辨率切片机制，更适合低质量或高密度文字影像

- **与第2讲 CLIP 的本质差异**：CLIP 计算相似度分数，不生成文字；LLaVA-Next 将图像直接注入语言模型上下文，可生成任意长度的自由文本回答——一个模型同时完成文字识别、文物描述和档案编目

- **三项数字人文分析任务**：（1）文字识别与转录——OCR 式提取，标注不确定字符；（2）文化元素识别——从材质、形制、纹饰三维度结构化描述；（3）标准化档案描述——生成可直接入库的四字段元数据记录

- **面向第10讲 Capstone**：`analyze_dh_frame()` 函数提供了统一的视觉分析接口，可接入第5讲提取的视频帧列表，与第1讲的文本摘要、第3讲的术语解释共同构成多模态数字人文档案管线的视觉侧组件